In [2]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import subprocess
import warnings

warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package])

try:
    from astroquery.heasarc import Heasarc
    import astropy
    import scipy
    import matplotlib
except ImportError:
    print("[SYSTEM] Desplegando Astroquery...")
    install("astroquery")
    install("astropy")
    install("scipy")
    install("matplotlib")
    from astroquery.heasarc import Heasarc

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER FINAL (Direct Mission Dump)...")

# --- BLOQUE 1: EXTRACCIÓN SOBERANA (METODO CONFIRMADO) ---
heasarc = Heasarc()
TARGET_TABLE = 'fermilgrb'

print(f"[QUERY] Descargando catálogo maestro '{TARGET_TABLE}'...")

try:
    # Volvemos al método que SÍ trajo las 231 filas con todas las columnas en tu log anterior.
    # query_mission_list baja la tabla entera sin recortes espaciales.
    table = heasarc.query_object(object_name='*', mission=TARGET_TABLE)

    df = table.to_pandas()
    # Forzamos mayúsculas para evitar errores de tipeo
    df.columns = [c.upper() for c in df.columns]
    print(f"[SUCCESS] Tabla recuperada. Registros: {len(df)}")

except Exception as e:
    print(f"[CRITICAL] Error de conexión HEASARC: {e}")
    sys.exit()

# --- BLOQUE 2: MAPEO DE COLUMNAS (DATA HARDCODED) ---
# Usamos EXACTAMENTE los nombres que aparecieron en tu log de éxito.
print("[MAP] Mapeando variables físicas...")

# Verificación explícita de columnas antes de procesar
required_cols = ['REDSHIFT', 'EXT_EMISSION_MAX_ENE', 'EXT_EMISSION_MAX_ENE_T', 'NAME']
missing = [col for col in required_cols if col not in df.columns]

if missing:
    print(f"[ERROR] Faltan columnas críticas a pesar de usar el método correcto: {missing}")
    print(f"[DEBUG] Columnas disponibles: {list(df.columns)}")
    sys.exit()

# Extracción directa
raw_z = df['REDSHIFT']
# Energía máxima del fotón extendido (MeV)
raw_e = df['EXT_EMISSION_MAX_ENE']
# Tiempo de llegada de ese fotón (s)
raw_t = df['EXT_EMISSION_MAX_ENE_T']
raw_n = df['NAME']

# --- BLOQUE 3: FÍSICA DE ALTA ENERGÍA ---
# Convertimos MeV a GeV
raw_e_gev = raw_e / 1000.0

# Filtros de Calidad (Stricto Sensu)
# 1. Redshift > 0 (Distancia conocida)
# 2. Energía > 1 GeV (Para ver efectos cuánticos y limpiar ruido)
# 3. Tiempo definido (No NaN)
mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 1.0) & (np.isfinite(raw_t))

clean_z = raw_z[mask].values
clean_t = raw_t[mask].values
clean_e = raw_e_gev[mask].values
clean_names = raw_n[mask].values

print(f"[DATA] Eventos válidos para LIV (z>0, E>1GeV): {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Pocos datos con E > 1 GeV. Relajando filtro a 0.1 GeV...")
    mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 0.1) & (np.isfinite(raw_t))
    clean_z = raw_z[mask].values
    clean_t = raw_t[mask].values
    clean_e = raw_e_gev[mask].values
    clean_names = raw_n[mask].values
    print(f"[DATA] Eventos válidos (Relaxed): {len(clean_z)}")

# Cálculo K-Factor (Integral Cosmológica)
print("[PHYS] Integrando expansión del universo...")
def get_k(z):
    z_grid = np.linspace(0, z, 50)
    # H(z)/H0
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    # K_jacob_piran = integral (1+z)/H(z)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

if len(clean_z) > 0:
    K_factors = np.array([get_k(z) for z in clean_z])
    # Observable Clave: Retraso por unidad de energía (s/GeV)
    # tau_n = dt / E
    lags = clean_t / clean_e
else:
    K_factors = []
    lags = []

# --- BLOQUE 4: ANÁLISIS Y REGRESIÓN ---
def model(k, m, c):
    return m * k + c

m_fit, m_err = 0, 0

if len(K_factors) > 2:
    # Ajuste robusto
    try:
        popt, pcov = curve_fit(model, K_factors, lags)
        m_fit = popt[0]
        m_err = np.sqrt(np.diag(pcov))[0]
    except Exception as e:
        print(f"[WARN] Fallo en ajuste: {e}")

# --- BLOQUE 5: VISUALIZACIÓN CIENTÍFICA ---
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

if len(K_factors) > 0:
    # Scatter plot con color por energía
    sc = plt.scatter(K_factors, lags, c=clean_e, cmap='plasma', s=100, alpha=0.9, label='Datos Fermi-LAT')
    plt.colorbar(sc, label='Energía (GeV)')

    # Barras de error estimadas para visualización
    plt.errorbar(K_factors, lags, yerr=0.1*np.abs(lags)+0.5, fmt='none', ecolor='gray', alpha=0.5)

    if len(K_factors) > 2:
        x_plot = np.linspace(0, max(K_factors)*1.1, 50)
        plt.plot(x_plot, model(x_plot, m_fit, popt[1]), color='#00ffcc',
                 linestyle='--', lw=2, label=f'Ajuste LIV (m={m_fit:.2e})')

    plt.axhline(0, color='white', linestyle=':', label='Einstein (m=0)')

    # Etiquetas inteligentes
    for i, name in enumerate(clean_names):
        # Etiquetar solo si tiene alto retraso o alta energía
        if clean_e[i] > 5.0 or abs(lags[i]) > 2:
            try:
                n_str = name.decode('utf-8') if isinstance(name, bytes) else str(name)
            except: n_str = str(name)
            plt.annotate(f"{n_str}\n{clean_e[i]:.1f}GeV",
                         (K_factors[i], lags[i]), color='yellow', fontsize=9,
                         xytext=(5, 5), textcoords='offset points')

plt.xlabel(r'Factor de Distancia Cosmológica $K(z)$')
plt.ylabel(r'Retraso Normalizado $\tau_n = \Delta t / E$ (s/GeV)')
plt.title('Cazando Violación de Lorentz (Final)\nDatos: HEASARC fermilgrb [EXT_EMISSION]', color='white')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[RESULT] Pendiente LIV: m = {m_fit:.4e} +/- {m_err:.4e} s/GeV")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[SYSTEM] Iniciando Protocolo ASTRO-MINER FINAL (Direct Mission Dump)...
[QUERY] Descargando catálogo maestro 'fermilgrb'...
[CRITICAL] Error de conexión HEASARC: HeasarcClass.query_object() missing 1 required positional argument: 'object_name'
Traceback (most recent call last):
  File "/tmp/ipython-input-2864846812.py", line 40, in <cell line: 0>
    table = heasarc.query_object(mission=TARGET_TABLE)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/utils/decorators.py", line 143, in deprecated_func
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
TypeError: HeasarcClass.query_object() missing 1 required positional argument: 'object_name'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/t

TypeError: object of type 'NoneType' has no len()